In [23]:
import pandas as pd
import numpy as np
import math
# import sys
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt

In [24]:
df = pd.read_csv("iris.csv",names=["sepal_length","sepal_width","petal_length","petal_width","class"])
df.head()

,sepal_length,sepal_width,petal_length,petal_width,class
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [25]:
df["class"] = df["class"].map({'Iris-setosa':0, 'Iris-versicolor':1, 'Iris-virginica':2})
df.head()

,sepal_length,sepal_width,petal_length,petal_width,class
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [26]:
X = df.drop(columns=["class"]).values
y = df["class"].values

# X_train,X_test,y_train,y_test = train_test_split(X,test_size=0.2,random_state=42)
# splitting the dataset into training and testing data
np.random.seed(42)
shuffled_indices = np.random.permutation(len(X))
split_idx = int(len(X) * 0.8)

train_indices = shuffled_indices[:split_idx]
test_indices = shuffled_indices[split_idx:]

X_train, y_train = X[train_indices], y[train_indices]
X_test, y_test = X[test_indices], y[test_indices]


# Dictionaries to store our computed parameters for each class
priors = {}
means = {}
variances = {}

n_samples = len(X_train)

for c in range(3):
    data = X_train[y_train == c]
    
    # Calculate P(y) -> Prior probability
    priors[c] = len(data) / n_samples
    
    # Calculating Mean for each of the 4 features given this class
    means[c] = np.mean(data, axis=0)
    
    # Calculate Variance for each of the 4 features given this class
    # We add 1e-9 (epsilon) to prevent division by zero in the prediction step
    variances[c] = np.var(data, axis=0) + 1e-9

print("Training complete! Parameters extracted successfully.")

Training complete! Parameters extracted successfully.


In [27]:
# Predict on X_test
y_pred = []

for x in X_test:
    posteriors = []
    
    for c in range(3):
        # Start with the log of the prior probability: log P(y)
        prior_log = np.log(priors[c])
        
        # Calculate Gaussian likelihood for each feature: P(x_i | y)
        mean = means[c]
        var = variances[c]
        
        numerator = np.exp(-((x - mean) ** 2) / (2 * var))
        denominator = np.sqrt(2 * np.pi * var)
        likelihood = numerator / denominator
        
        #Sum the logs of the likelihoods to prevent floating-point underflow
        likelihood_log_sum = np.sum(np.log(likelihood))
        
        #Total posterior probability score
        posterior = prior_log + likelihood_log_sum
        posteriors.append(posterior)

    y_pred.append(np.argmax(posteriors))

# Evaluation of the model
y_pred = np.array(y_pred)
accuracy = np.mean(y_pred == y_test)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

Test Accuracy: 93.33%
